# Topic Modeling With BERTopic

* * * 

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Use BERTopic to group r/AmItheAsshole submissions by the issues people write about.
* Interpret and visualize the topics found by the model.
* Try out ways to improve or simplify the model when topics overlap too much or don’t make sense.
* Practice naming and describing topics, and using them to organize or classify new text.
</div>

### Icons Used in This Notebook
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.<br>

### Sections
1. [Topic Modeling with BERTopic](#topic)
2. [Explore Selected Topics](#explore)
3. [Tuning our Model](#tune)
4. [Finding Representative Posts](#repr)
5. [Doing Things With Topic Models](#doing)


<a id='topic'></a>

# Topic Modeling with BERTopic

In this notebook, we explore **BERTopic**, a topic modeling tool that leverages BERT embeddings and [c-TF-IDF](https://maartengr.github.io/BERTopic/api/ctfidf.html), to extract topics.


## Use Cases for Topic Modeling

A. Exploratory analysis:
- “What are customers complaining about most?”
- “What themes exist in 80,000 survey responses?”
- “What are the policy concerns in legislative text?”
- “What are the main themes in employee comments?”

B. Downstream prediction tasks:
- churn prediction
- fraud detection
- political ideology prediction
- etc.

C. Trend analysis over time:
- topic prevalence across months/years
- detecting emerging themes

D. Search, retrieval, routing:
- content tagging
- filters for helpdesk tickets
- Q&A automation routing
- recommended reading / “more like this” suggestions

E. Audit, compliance, risk monitoring:
- trace harmful content
- emergent risk clusters
- compliance breaches in text
- safety + moderation issues

## About BERT Embeddings
BERT embeddings are vector representations of text created by the BERT model, which is a large transformer-based neural network trained to understand language context. Unlike traditional word embeddings (like word2vec), BERT creates contextual embeddings. That means the same word will have different vector representations depending on its surrounding words. BERT processes entire sentences at once, using its attention mechanism to capture meaning and relationships between words.

BERTopic works as follows:

1. **Understands text using BERT:**  <br> 
Instead of just counting words, BERTopic uses a BERT-like Sentence Transformer model (e.g. "all-MiniLM-L6-v2")) to turn each post (up to 512 tokens, to be precise) into a “vector”—a list of numbers that captures the meaning and context of a post. This makes it better at grouping together texts that talk about similar things, even if they use different words. BERT-like models were trained on natural text, including punctuation, function words, and syntactic structure. When we are training the model on our reddit posts, the model:
   - sends the entire document string (the whole Reddit post) to the embedding model.
   - truncates the text at its maximum sequence length (typically 512 tokens).
   - encodes the text internally into token-level embeddings (usually of 384 dimensions).
   - averages (or otherwise pools) them → one dense vector summarizing meaning. That vector represents your entire post in semantic space.

2. **Reduces dimensions:** <br>
BERTopic then uses an algorithm called **UMAP** to reduce these learned embedding dimensions (usually from 384 to 5). It is basically creating a compact version of the embedding space where meaningful clusters can actually emerge.

3. **Clusters similar documents:**  <br> 
BERTopic then looks for clusters (groups) of documents that are close to each other in this vector space. It uses an algorithm called **HDBSCAN** that decides, based on the data, how many clusters (topics) make sense. This means you don’t have to guess the number of topics in advance.

4. **Topic representation (labeling):** <br>
Now that clusters exist, BERTopic gathers all documents in each cluster. It applies class-based TF-IDF (c-TF-IDF) to those clusters of texts to count which words/phrases are most frequent. It uses those counts to label or “describe” each topic with top words. c-TF-IDF treats each cluster (topic) as a "document".


5. **Visualizes topics and documents:**   <br>
BERTopic comes with interactive tools to show you how your topics relate to each other, how common each topic is, and where your documents fit in.

See the [documentation](https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html) to learn more about this pipeline.

## IMPORTANT: Goal of This Notebook

The main goal of this notebook is to see how BERTopic works, but also to understand the techniques that go into creating interpretable topics. 

This is a **highly iterative process** that will depend on the dataset you are working with. You should expect to spend several hours tuning and your model. This is **NOT** a one-shot / "just run the code" process.

My hope is that you will be able to see the levers you have at your disposal to improve your BERTopic models -- and that often, there is no such thing as a silver bullet. 


### Note on package installation
This cell makes sure all the Python packages needed for this lesson are installed.
Running this cell will check for each package and install it if it’s missing, so your notebook runs smoothly.

In [ ]:
# restart kernel after running
#%pip install bertopic 

⚠️ **Warning:** This notebook might run slowly on your local machine depending on your system architecture. For faster execution, open it in Google Colab and select a GPU runtime.

## Loading the Data

In [ ]:
import pandas as pd

# Load your preprocessed AITA CSV (change the filename if needed)
df = pd.read_csv('../../data/aita_top_subs.csv')

In [ ]:
df = df[~df['selftext'].isin(['[deleted]', '[removed]'])].reset_index(drop=True)
df.shape

In [ ]:
import re

def light_clean(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['selftext_clean'] = df['selftext'].apply(light_clean)
df = df[df['selftext_clean'].str.split().apply(len) > 30]

In [ ]:
docs = df['selftext_clean'].tolist()

## Build and Fit BERTopic Model

💡 **Tip**: The output of this model is **stochastic** because UMAP’s dimensionality reduction, HDBSCAN’s clustering, and random initialization in the embedding and vectorization steps each introduce small elements of randomness. That menas running the same code again may yield slightly different topic groupings and word distributions than those shown here.

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(verbose=True, embedding_model="all-MiniLM-L6-v2", ) # this is the default model

In [ ]:
# converting the docs into embeddings, reducing dimensions, clustering, labeling
topics, probabilities = topic_model.fit_transform(docs)

<a id='explore'></a>

# Explore Extracted Topics

Let's view topic frequencies and the top words per topic.

In [ ]:
topic_model.get_topic_info()


### About Topic -1

The first row, Topic -1, in BERTopic is a “catch-all” for documents that don’t fit into any meaningful cluster.

Here's how that works:
- HDBSCAN (the clustering algorithm) automatically labels “noise” or “outlier” documents with -1.
- These are typically posts that are too unique, too generic, or just don’t belong to any clear topic group.
- Including topic -1 in your list of topics will show a “topic” that’s not really coherent, and the top words for -1 are usually either very generic or meaningless.
- Most users ignore topic -1 when reviewing topics and top words, focusing only on the numbered topics (0, 1, 2, …).

There are many posts in this -1 cluster (almost half of our ~8000 posts!), which means we cannot use them for further analysis.

Typical and healthy ranges:
- 10–25% outliers: usually indicates a good, interpretable model.
- Below 5%: might suggest over-clustering (too few distinct topics or too broad topics).
- Above 40%: often means your embeddings are too sparse or the clustering parameters are too strict.



### Look at individual topics 
The `get_topic()` method allows us to see the top words for a topic. These top words should be interpretable; that is, collectively they should point to a topic. If they don't, you have more preprocessing and finetuning to do.

In [ ]:
# Show top words for topic 1
topic_model.get_topic(1)

As you can see, this topic (like many others) is full of function words. 

### Look at topics per document
BERTopic can also take our input documents (i.e., our Reddit posts in df['selftext']) and return a DataFrame that maps each document → its topic assignment + metadata:

In [ ]:
topic_model.get_document_info(df['selftext'])

### Intertopic Distance Map
The `visualize_topics` function visualizes topics and their similarity in an interactive plot. We're also saving it to disk so it can be embedded on a website.

This is a **very helpful** visualization. In this 2D space, we ideally want to see topics that are non-overlapping, and spread across the space. 

In [ ]:
fig = topic_model.visualize_topics()
fig.show()

<a id='tune'></a>

# Tuning our Model

Common issues when you build your first topic model:

1. **Vocabulary**. It seems our topics are not very interpretable -- they have lots of function words like and, to, my, her, the, she etc. We need to control the vocabulary. We could consider removing stopwords, punctuation, or lemmatizing. However, if we do that in our preprocessing function, we could actually hurt topic quality, because it disrupts the contextual flow the BERT model was trained on. 

2. **Outliers**. We have lots of outlier documents, which have not been classified usefully at all. Reducing the number of -1 outliers is not always the goal — it’s more about balance.

3. **Overlap**. Our intertopic distance map shows lots of overlapping bubbles, meaning our model has produced **too many fine-grained topics**. This is common. Similar documents can get split into clusters that aren't really distinct.

BERTopic allows us to change some **hyperparameters** at different steps of the model to improve it. Let's see what we can do to try and ameliorate these issues.

### 1. Controlling Vocabulary: Add a custom vectorizer

Second, to prevent that topic keywords are full of stopwords and random filler, we can give BERTopic a better word list to use when running TF-IDF. We can do that by passing in a custom `CountVectorizer` that BERTopic uses to build its c-TF-IDF on. 

We can remove some common stopwords, as well as a hand-picked list of common words from this corpus. This is an iterative process: have a look at your topics, figure out which low-information terms appear in all topics (or one huge topic), and add them to this list. 

Note: this CountVectorizer runs *after* creating the BERT embeddings model, reducing dimensionality, and clustering it with HDBSCAN. This means the model still learns good representations from the natural input text -- we simply filter the words shown in these topics.

The result we expect from this: cleaner topic labels, and more interpretable word clusters.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords

stopwords_nltk = stopwords.words('english')
aita_stopwords = stopwords_nltk + [
    'aita', 'yta', 'nta', 'esh', 'info', 'asshole', 'op',
    'edit', 'update', 'post', 'story',
    'please', 'thank', 'thanks', 'hi', 'hello',
    'said', 'told', 'like', 'would', 'get',
    'got', 'time','one','want','really','asked','know',
    'thing','things','feel','felt','make','made', 'since', 'go', 'going'
]

# we will also include bigram tokens
vectorizer = CountVectorizer(stop_words=aita_stopwords,
                             ngram_range=(1, 2))

We will pas this `vectorizer` into the model when we retrain it.

In [ ]:
topic_model_v2 = BERTopic(embedding_model="all-MiniLM-L6-v2", vectorizer_model=vectorizer, 
                          calculate_probabilities=True, verbose=True) 

In [ ]:
# converting the docs into embeddings, reducing dimensions, clustering, labeling
topics_v2, probabilities_v2 = topic_model_v2.fit_transform(docs)

In [ ]:
# have a look again at how many topics we have, and how big our outlier topic is
topic_model_v2.get_topic_info()

### 2. Reduce the Number of Topics

To prevent redundant and overlapping clusters, we can merge semantically similar topics using BERTopic’s internal topic similarity.
We can do this with the `reduce_topics` method:

In [ ]:
# Reduce the number of topics (set to the number you want)
topic_model_v2.reduce_topics(docs, nr_topics=30)

In [ ]:
# look again at the topic info
topic_model_v2.get_topic_info()

In [ ]:
fig = topic_model_v2.visualize_topics()
fig.show()

Zoom in on the graph to see different clusters of topics.

### 4. Hyperparameter Tuning

There are many more things we can do to try and improve our model:

- We can change the minimum amount of documents a topic contains with the `min_topic_size` hyperparameter in BERTopic.
- We can adjust HDBSCAN’s hyperparameters, such as the minimum cluster size using the `min_cluster_size` hyperparameter to allow smaller, more granular clusters.
- We can change the hyperparameters for UMAP.
- We can try a different embedding model like `all-distilroberta-v1`.

⚠️ **Warning:** While these steps can help reduce outliers, they also to an explosion of topics. The goal is to strike a balance.

Let's try and change some hyperparameters now. Expect this to take longer than our initial run!


In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Dimensionality reduction
# for more info on these parameters, see https://hdbscan.readthedocs.io/en/latest/api.html 

umap_model = UMAP(
    n_neighbors=15,       # local vs global structure
    n_components=10,      # dimensionality of reduced space
    min_dist=0.0,         # tighter clusters
    metric="cosine",
    random_state=42
)

# Density-based clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  # how many docs per topic (granularity)
    min_samples=2,        # how strict the clustering is
    prediction_data=True,
    cluster_selection_method="eom" # how clusters are calculated -> "leaf" results in more granular clusters 
)

# Topic model
topic_model_v3 = BERTopic(
    embedding_model="all-distilroberta-v1",  # can also try another model, e.g. all-mpnet-base-v2
    vectorizer_model=vectorizer,          # custom stopwords here
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    min_topic_size=20,                    # smallest amount of document allowed in a topic -> smaller number means more granular topics
    calculate_probabilities=True,
    verbose=True
)

topics_v3, probabilities_v3 = topic_model_v3.fit_transform(docs)

In [ ]:
topic_model_v3.get_topic_info()

Let's reduce our topics again:

In [ ]:
# Reduce the number of topics (set to the number you want)
topic_model_v3.reduce_topics(docs, nr_topics=50)

In [ ]:
topic_info_v3 = topic_model_v3.get_topic_info()
topic_info_v3

In [ ]:
# Re-visualize the intertopic distance map
fig = topic_model_v3.visualize_topics()
fig.show()

## Picking a Model

Let's pick our tuned model going forward for now. It has a lot of outliers still.

In your own data, if you want to use your topic modeling labels for downstream analysis, you will want to try and reduce those outliers -- otherwise you won't be able to use these posts!

### Visualizing all docs
We can use the `visualize_documents` method to visualize our posts in 2d space see the clustering of our chosen topic model. IF you hover over the dots, you will see the associated selftext. This is another way to visually inspect whether your topic model makes sense.

Note that the clusters are labeled with their number and the top words in the cluster.
 
⚠️ **Warning:** Rendering this visualization might take a while!

In [ ]:
topic_model_v3.visualize_documents(docs, topics=topics_v3)

Let's say I'm interested in topic 4.

**⚠️ Warning:** If you run this code again your topics might look different due to the probabilistic nature of UMAP. 

In [ ]:
docs_per_topic = topic_model_v3.get_representative_docs()
docs_per_topic[4]


### Grab all posts from a certain topic (for further processing)

Once you’ve trained a BERTopic model, each document is assigned a dominant topic. You can use this assignment to extract all posts that belong to a specific topic for closer analysis, visualization, or downstream tasks.

For instance, let's create a new dataframe with all posts that have topic 4 as the dominant topic.

First, let's add topics and probabilities to our original DataFrame.

In [ ]:
import pandas as pd

# Add dominant topic per document
df["dominant_topic"] = topics_v3

# Optionally: add the highest probability (confidence)
import numpy as np
df["topic_confidence"] = np.max(probabilities_v3, axis=1)

In [ ]:
# Example: compare topic 4 vs topic 7
df_topic4 = df[df["dominant_topic"] == 4].copy()
df_topic7 = df[df["dominant_topic"] == 7].copy()


<a id="doing"></a>

# Doing Things With Topic Models 

### Compare Sentiment Between Topics

We can apply this set of topic categories -- basically another feature in our dataset -- to downstream tasks. For instance, we can compare average sentiments between two interesting topics.

We can use the transformers pipeline from Hugging Face -- a modern approach that:

- uses RoBERTa, trained on millions of social-media posts.
- quickly classifies each post as positive / neutral / negative.
- produces a small comparison table showing sentiment ratios per topic.

In [ ]:
from transformers import pipeline

# Load a modern sentiment model
sentiment_analyzer = pipeline("sentiment-analysis",
                              model="cardiffnlp/twitter-roberta-base-sentiment-latest")

# Apply it (keep short for this demo)
df_topic4["sentiment"] = df_topic4["selftext"].head(50).apply(lambda x: sentiment_analyzer(x[:512])[0]["label"])
df_topic7["sentiment"] = df_topic7["selftext"].head(50).apply(lambda x: sentiment_analyzer(x[:512])[0]["label"])


In [ ]:
# Compare proportions
comparison = (
    pd.concat([
        df_topic4.assign(topic="Topic 4"),
        df_topic7.assign(topic="Topic 7")
    ])
    .groupby(["topic", "sentiment"])
    .size()
    .unstack(fill_value=0)
    .apply(lambda x: x / x.sum(), axis=1)
)

print(comparison)

In [ ]:
import matplotlib.pyplot as plt

# Plot as stacked bar chart
comparison.plot(kind="bar", stacked=True, colormap="coolwarm", figsize=(6,4))
plt.title("Sentiment Distribution by Topic")
plt.xlabel("Topic")
plt.ylabel("Proportion")
plt.legend(title="Sentiment", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Visualizing Topics Over Time

BERTopic can show you how topics change over time. This is helpful for understanding trends, seasonal patterns, or how certain topics gain or lose popularity in your dataset.

To do this, we need:
- Our list of documents
- The topics assigned to each document
- A timestamp for each document

### Converting Timestamps

First, we need to convert the Unix timestamps in the `created` column to readable dates. Unix timestamps are numbers representing seconds since January 1, 1970.

In [ ]:
# Convert Unix timestamp to datetime
df['timestamp'] = pd.to_datetime(df['created'], unit='s')

# Look at the first few timestamps
df['timestamp'].head()

### Creating the Topics Over Time Visualization

Now we can use BERTopic's `topics_over_time()` function to calculate how each topic appears across different time periods. This function groups documents by time bins and counts topic frequencies.

In [ ]:
# Ensure timestamps are clean and properly formatted
# We do this once here, not at the very start of your notebook, 
# so we don't drop any rows that might still be useful for modeling
df = df.dropna(subset=["timestamp"]).reset_index(drop=True)
timestamps = pd.to_datetime(df["timestamp"]).dt.to_pydatetime().tolist()

# Verify your topic assignments are valid after reduction.
# Sometimes reduce_topics renumbers or merges topics.
valid_topics = set(topic_model_v3.get_topic_info()["Topic"])
topics_v3_clean = [t if t in valid_topics else -1 for t in topics_v3]

# Compute topic dynamics over time.
topics_over_time = topic_model_v3.topics_over_time(
    docs=docs,
    topics=topics_v3_clean,
    timestamps=timestamps,
    nr_bins=20,          # number of time bins to group posts
    global_tuning=False  # avoid index errors with reduced topics
)

In [ ]:
# Interactive visualization using Plotly
topic_model_v3.visualize_topics_over_time(topics_over_time)

### Interpreting the Visualization

The plot above shows how frequently each topic appears over time. Each line represents a different topic, and you can:

- **See trends**: Which topics became more or less popular over time?
- **Identify spikes**: Were certain topics only discussed during specific periods?
- **Compare topics**: Which topics dominate your dataset at different points in time?

💡 **Tip**: You can hover over the lines to see the topic number and frequency at specific dates. You can also click on topic numbers in the legend to show/hide specific topics.